# 07 — QAT → TRT Inference

Three-step pipeline for each `(PRECISION, input_bits)` pair:

1. **Export** — load QAT checkpoint, export to QDQ ONNX → `onnx/qat/<prec>_in<N>b/resnet18_qat_<prec>_qdq.onnx`
2. **Build engine** — compile TRT engine from QAT QDQ ONNX → `engines/qat/<run_id>.engine`
3. **Inference** — run `trt_evaluate`, save → `runs/qat_trt/<run_id>/result.json`

Each step skips if the target file already exists.

**Prerequisite:** QAT training must have run and produced checkpoints under `checkpoints/qat/<prec>_in<N>b/`.

## Setup

In [ ]:
from pathlib import Path
import sys, json, time

sys.path.insert(0, str(Path("../src").resolve()))
sys.path.insert(0, str(Path("../src/qat_modelopt").resolve()))

import onnx
import pandas as pd
import torch
import torch.nn as nn

from config import ExperimentConfig, set_seed
from data import build_runner_loaders
from model import ResNet18
from onnx_exporter import ONNXExporter
from quantize import restore_modelopt_state
from trt_builder import build_engine
from trt_infer import trt_evaluate

pd.set_option("display.max_columns", 200)

## Configuration

In [ ]:
REPO             = Path("..").resolve()
CKPT_ROOT        = REPO / "checkpoints" / "qat"
ONNX_ROOT        = REPO / "onnx" / "qat"
ENGINE_ROOT      = REPO / "engines" / "qat"
RUNS_ROOT        = REPO / "runs" / "qat_trt"

PRECISION        = "int8"          # "int8" or "int4"
INPUT_BITS_LIST  = [1, 2, 4, 8]
NUM_CLASSES      = 100
BATCH_SIZE       = 1
NUM_WORKERS      = 8
NUM_EVAL_BATCHES = 500             # set to None for full val set
SEED             = 42
DEVICE           = "cuda" if torch.cuda.is_available() else "cpu"

print(f"checkpoints : {CKPT_ROOT}")
print(f"onnx out    : {ONNX_ROOT}")
print(f"engines out : {ENGINE_ROOT}")
print(f"results     : {RUNS_ROOT}")
print(f"device      : {DEVICE}")

## Helpers

In [ ]:
def qat_trt_run_id(precision: str, bits: int, batch_size: int, device: str) -> str:
    return f"resnet18_qat_trt_{precision}_in{bits}b_{device.split(':')[0]}_bs{batch_size}"


def qat_onnx_path(precision: str, bits: int) -> Path:
    return ONNX_ROOT / f"{precision}_in{bits}b" / f"resnet18_qat_{precision}_qdq.onnx"


def qat_engine_path(run_id: str) -> Path:
    return ENGINE_ROOT / f"{run_id}.engine"


def load_qat_model(ckpt_dir: Path, num_classes: int) -> nn.Module:
    """Load QAT ResNet-18: restore graph first, then weights."""
    pth_path = ckpt_dir / "qat_modelopt_best.pth"
    mo_path  = ckpt_dir / "qat_modelopt_best_mostate.pt"
    if not pth_path.exists() or not mo_path.exists():
        raise FileNotFoundError(f"Missing checkpoint pair in {ckpt_dir}")
    model = ResNet18(num_classes=num_classes, pretrained=False)
    restore_modelopt_state(model, str(mo_path))
    ckpt = torch.load(pth_path, map_location="cpu")
    model.load_state_dict(ckpt["model"])
    return model.eval()


def save_result(payload: dict, out_dir: Path) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / "result.json"
    path.write_text(json.dumps(payload, indent=2, sort_keys=True))
    return path

## Step 1 — Export QAT model to QDQ ONNX

The QAT model has `TensorQuantizer` modules inserted by ModelOpt. `torch.onnx.export`
maps these to `QuantizeLinear` / `DequantizeLinear` nodes automatically, producing a
QDQ-annotated ONNX with QAT-trained scales — no PTQ calibration needed.

In [ ]:
for bits in INPUT_BITS_LIST:
    onnx_path = qat_onnx_path(PRECISION, bits)
    ckpt_dir  = CKPT_ROOT / f"{PRECISION}_in{bits}b"

    if onnx_path.exists():
        print(f"[skip] ONNX exists: {onnx_path}")
        continue

    if not ckpt_dir.exists():
        print(f"[skip] No checkpoint at {ckpt_dir}")
        continue

    print(f"\n=== Exporting {PRECISION}_in{bits}b ===")
    model = load_qat_model(ckpt_dir, NUM_CLASSES)
    onnx_path.parent.mkdir(parents=True, exist_ok=True)
    ONNXExporter(model, "cpu", onnx_path).export_model(
        opset_version=17,
        dynamic_batch=True,
    )

    m   = onnx.load(str(onnx_path))
    ops = [n.op_type for n in m.graph.node]
    print(f"  Q: {ops.count('QuantizeLinear')}, DQ: {ops.count('DequantizeLinear')}")
    print(f"  Saved → {onnx_path}")

## Step 2 — Build TRT engines

In [ ]:
for bits in INPUT_BITS_LIST:
    run_id      = qat_trt_run_id(PRECISION, bits, BATCH_SIZE, DEVICE)
    onnx_path   = qat_onnx_path(PRECISION, bits)
    engine_path = qat_engine_path(run_id)

    if engine_path.exists():
        print(f"[skip] Engine exists: {engine_path}")
        continue

    if not onnx_path.exists():
        print(f"[skip] ONNX missing — run export step first: {onnx_path}")
        continue

    print(f"\n=== Building engine: {run_id} ===")
    ENGINE_ROOT.mkdir(parents=True, exist_ok=True)
    build_engine(
        onnx_path   = onnx_path,
        engine_path = engine_path,
        precision   = PRECISION,
        batch_size  = BATCH_SIZE,
    )
    print(f"  Engine → {engine_path}")

## Step 3 — TRT Inference

In [ ]:
records = []

for bits in INPUT_BITS_LIST:
    run_id      = qat_trt_run_id(PRECISION, bits, BATCH_SIZE, DEVICE)
    engine_path = qat_engine_path(run_id)

    if not engine_path.exists():
        print(f"[skip] Engine missing — run build step first: {engine_path}")
        continue

    print(f"\n=== {run_id} ===")

    cfg = ExperimentConfig(
        backend          = "tensorrt",
        device           = DEVICE,
        model_precision  = PRECISION,
        batch_size       = BATCH_SIZE,
        num_workers      = NUM_WORKERS,
        input_quant_bits = bits,
        num_classes      = NUM_CLASSES,
        num_eval_batches = NUM_EVAL_BATCHES,
        seed             = SEED,
    )
    set_seed(cfg)

    _, val_loader = build_runner_loaders(cfg)
    criterion     = nn.CrossEntropyLoss()

    t0      = time.perf_counter()
    tracker = trt_evaluate(engine_path, cfg, val_loader, criterion)

    payload = {
        "status":  "ok",
        "run_id":  run_id,
        "system":  cfg.stamp(),
        "config":  {
            **cfg.to_dict(),
            "qat_checkpoint_dir": str(CKPT_ROOT / f"{PRECISION}_in{bits}b"),
            "qat_onnx_path":      str(qat_onnx_path(PRECISION, bits)),
        },
        "results": tracker.summary(),
        "error":   None,
        "total_eval_time_sec": round(time.perf_counter() - t0, 3),
    }
    saved = save_result(payload, RUNS_ROOT / run_id)
    print(f"  saved → {saved}")
    records.append(payload)

## Summary

In [ ]:
rows = []
for r in records:
    res = r["results"]
    rows.append({
        "run_id":         r["run_id"],
        "input_bits":     r["config"]["input_quant_bits"],
        "precision":      r["config"]["model_precision"],
        "top1_acc":       res["top1_acc"],
        "top5_acc":       res["top5_acc"],
        "infer_ms_avg":   res["infer_ms_avg"],
        "throughput_sps": res["throughput_infer_sps"],
        "total_samples":  res["total_samples"],
    })

df = pd.DataFrame(rows).sort_values("input_bits").reset_index(drop=True)
df